# IEEE 118-bus DP3Ph and EMT3Ph comparison

This notebook builds the IEEE 118-bus network directly in Python from the MATPOWER case data, solves one common power flow, and initializes matching 4th-order SG models in DP3Ph and EMT3Ph. Both runs cover 30 seconds and compare voltage envelopes, startup consistency, boundedness, and runtime.

In [ ]:
from pathlib import Path
import os, re, sys, tempfile, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import savemat

_here = Path.cwd().resolve()
REPO = next(
    p for p in [_here, *_here.parents] if (p / "python" / "src" / "dpsim").is_dir()
)
os.chdir(REPO)
sys.path.insert(0, str(REPO / "build"))
sys.path.insert(0, str(REPO / "python" / "src"))
import dpsimpy

# Preserve the actual EMT three-phase switch before MATPOWER configures its aliases.
EMT_PH3_SWITCH = dpsimpy.emt.ph3.Switch
DP3PH_SWITCH = dpsimpy.dp.ph3.Switch
import importlib
from dpsim import matpower as _matpower

matpower = importlib.reload(_matpower)
assert (
    Path(matpower.__file__).resolve()
    == (REPO / "python" / "src" / "dpsim" / "matpower.py").resolve()
), matpower.__file__

CASE_FILE = REPO / "dpsim" / "examples" / "matpower" / "case118.m"
FREQUENCY = 60.0
BASE_MVA = 100.0

## Load the canonical case and assign shared SG4 data

The network tables come from the checked-in MATPOWER case118 source. Since MATPOWER does not define dynamic machine data, each listed synchronous generator is assigned the same reasonable 4th-order parameters on its listed MVA base. DP and EMT use the same data.

In [ ]:
case_text = CASE_FILE.read_text()


def parse_matrix(name):
    match = re.search(rf"mpc\.{name}\s*=\s*\[(.*?)\];", case_text, re.S)
    if not match:
        raise ValueError(f"Missing MATPOWER table {name}")
    rows = []
    for line in match.group(1).splitlines():
        line = line.split("%", 1)[0].strip().rstrip(";")
        if line:
            rows.append(np.fromstring(line, sep=" "))
    return np.vstack(rows)


BUS, GEN, BRANCH = (parse_matrix(k) for k in ("bus", "gen", "branch"))
assert BUS.shape == (118, 13) and len(GEN) == 54 and len(BRANCH) == 186
# gen_dyn columns: bus, model, BaseS, H, Ra, Xl, Td0_t, Td0_s, Tq0_t, Tq0_s, Xd, Xd_t, Xd_s, Xq, Xq_t, Xq_s
DYN = np.array(
    [
        [
            g[0],
            4,
            max(g[6], g[8]),
            4.0,
            0.002,
            0.15,
            5.0,
            1.0,
            0.5,
            0.05,
            1.75,
            0.425,
            0.3,
            1.75,
            0.425,
            0.3,
        ]
        for g in GEN
    ],
    dtype=float,
)
TEMP = Path(tempfile.mkdtemp(prefix="ieee118_dpsim_"))
STATIC_PATH, DYNAMIC_PATH = TEMP / "case118.mat", TEMP / "case118_dyn.mat"
savemat(
    STATIC_PATH,
    {"mpc": {"baseMVA": BASE_MVA, "bus": BUS, "gen": GEN, "branch": BRANCH}},
)
savemat(DYNAMIC_PATH, {"mpc": {"gen_dyn": DYN}})
print(
    f"Loaded IEEE118: {len(BUS)} buses, {len(GEN)} synchronous generators, {len(BRANCH)} branches"
)
print(
    f"Shared SG4 parameters: H={DYN[0,3]:g}s, Xd={DYN[0,10]:g}, Xd'={DYN[0,11]:g}, Xq={DYN[0,13]:g}, Xq'={DYN[0,14]:g}"
)

## Common power flow

The power-flow solution is built once and transferred into both dynamic topologies.

In [ ]:
pf_reader = matpower.Reader(str(STATIC_PATH))
pf_reader.create_dpsim_objects(
    matpower.Domain.PF,
    FREQUENCY,
    log_level=dpsimpy.LogLevel.off,
    filter_out_of_service=True,
)
pf_reader.create_dpsim_topology()
pf = dpsimpy.Simulation("IEEE118_SP_PF")
pf.set_system(pf_reader.system)
pf.set_domain(dpsimpy.Domain.SP)
pf.set_solver(dpsimpy.Solver.NRP)
pf.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Initialization)
pf.do_init_from_nodes_and_terminals(False)
pf.set_time_step(0.01)
pf.set_final_time(0.01)
pf.run()
print("Solved common PF for", len(pf_reader.dpsimpy_busses_dict), "buses")

## 30-second DP3Ph run

In [ ]:
dp_reader = matpower.Reader(str(STATIC_PATH), mpc_dyn_file_path=str(DYNAMIC_PATH))
dp_reader.create_dpsim_objects(
    matpower.Domain.DP3PH,
    FREQUENCY,
    log_level=dpsimpy.LogLevel.off,
    with_pss=False,
    with_avr=False,
    with_tg=False,
    filter_out_of_service=True,
    generator_model=matpower.GenModel.SG_4ORDER_VBR,
)
dp_reader.create_dpsim_topology()
dp_reader.system.init_with_powerflow(
    systemPF=pf_reader.system, domain=dpsimpy.Domain.DP
)
dp_logger = dpsimpy.Logger("IEEE118_DP3Ph")
PLOT_BUSES = [1, 10, 30, 69, 100, 118]
for bus_id in PLOT_BUSES:
    dp_logger.log_attribute(
        f"N{bus_id}.V", "v", dp_reader.dpsimpy_busses_dict[f"N{bus_id}"]
    )
for gen_bus in (10, 69, 89):
    dp_logger.log_attribute(
        f"Gen_N{gen_bus}.w_r", "w_r", dp_reader.dpsimpy_comp_dict[f"Gen_N{gen_bus}"][0]
    )
dp_sim = dpsimpy.Simulation("IEEE118_DP3Ph")
dp_sim.set_system(dp_reader.system)
dp_sim.set_domain(dpsimpy.Domain.DP)
dp_sim.set_solver(dpsimpy.Solver.MNA)
dp_sim.do_init_from_nodes_and_terminals(True)
dp_sim.set_time_step(1e-3)
dp_sim.set_final_time(30.0)
dp_sim.add_logger(dp_logger)
t0 = time.perf_counter()
dp_sim.run()
dp_runtime = time.perf_counter() - t0
dp_trace = pd.read_csv(REPO / "logs" / "IEEE118_DP3Ph.csv", skipinitialspace=True)
print(
    f"DP: {len(dp_trace)} samples, {dp_trace.time.iloc[-1]:.3f}s, {dp_runtime:.2f}s, finite={np.isfinite(dp_trace.iloc[:,1:].to_numpy()).all()}"
)

## 30-second EMT3Ph run

In [ ]:
emt_reader = matpower.Reader(str(STATIC_PATH), mpc_dyn_file_path=str(DYNAMIC_PATH))
emt_reader.create_dpsim_objects(
    matpower.Domain.EMT,
    FREQUENCY,
    log_level=dpsimpy.LogLevel.off,
    with_pss=False,
    with_avr=False,
    with_tg=False,
    filter_out_of_service=True,
    generator_model=matpower.GenModel.SG_4ORDER_VBR,
)
emt_reader.create_dpsim_topology()
emt_reader.system.init_with_powerflow(
    systemPF=pf_reader.system, domain=dpsimpy.Domain.EMT
)
emt_logger = dpsimpy.Logger("IEEE118_EMT3Ph", True, 2)
for bus_id in PLOT_BUSES:
    emt_logger.log_attribute(
        f"N{bus_id}.V", "v", emt_reader.dpsimpy_busses_dict[f"N{bus_id}"]
    )
for gen_bus in (10, 69, 89):
    emt_logger.log_attribute(
        f"Gen_N{gen_bus}.w_r", "w_r", emt_reader.dpsimpy_comp_dict[f"Gen_N{gen_bus}"][0]
    )
emt_sim = dpsimpy.Simulation("IEEE118_EMT3Ph")
emt_sim.set_system(emt_reader.system)
emt_sim.set_domain(dpsimpy.Domain.EMT)
emt_sim.set_solver(dpsimpy.Solver.MNA)
emt_sim.do_init_from_nodes_and_terminals(True)
emt_sim.set_time_step(5e-4)
emt_sim.set_final_time(30.0)
emt_sim.add_logger(emt_logger)
t0 = time.perf_counter()
emt_sim.run()
emt_runtime = time.perf_counter() - t0
emt_trace = pd.read_csv(REPO / "logs" / "IEEE118_EMT3Ph.csv", skipinitialspace=True)
print(
    f"EMT: {len(emt_trace)} samples, {emt_trace.time.iloc[-1]:.3f}s, {emt_runtime:.2f}s, finite={np.isfinite(emt_trace.iloc[:,1:].to_numpy()).all()}"
)
print(f"Runtime ratio EMT/DP: {emt_runtime/dp_runtime:.2f}x")

## Initialization, bounds, and voltage comparison

The EMT traces are converted to rolling phase RMS values before comparison with the DP phasor magnitudes. The initial voltage comparison uses the first output samples; per-bus min/max checks cover the full 30-second simulation.

In [ ]:
def dp_voltage_pu(trace, bus_id):
    re = trace[f"N{bus_id}.V_1.re"].to_numpy()
    im = trace[f"N{bus_id}.V_1.im"].to_numpy()
    base_kv = BUS[int(bus_id) - 1, 9]
    return np.hypot(re, im) / (base_kv * 1e3 * np.sqrt(2 / 3))


def emt_voltage_pu(trace, bus_id, dt=None):
    if dt is None:
        dt = float(np.median(np.diff(trace.time.to_numpy())))
    cols = [f"N{bus_id}.V_{p}" for p in range(3)]
    phase_rms = np.sqrt((trace[cols].to_numpy() ** 2).mean(axis=1))
    base_kv = BUS[int(bus_id) - 1, 9]
    phase_rms_base = base_kv * 1e3 / np.sqrt(3)
    window = max(2, round(1.0 / FREQUENCY / dt))
    return pd.Series(
        np.sqrt(
            pd.Series((phase_rms / phase_rms_base) ** 2)
            .rolling(window, min_periods=window)
            .mean()
        ),
        index=trace.index,
    )


fig, axes = plt.subplots(len(PLOT_BUSES), 1, figsize=(12, 13), sharex=True)
summary = []
for ax, bus_id in zip(axes, PLOT_BUSES):
    dp_v = dp_voltage_pu(dp_trace, bus_id)
    emt_v = emt_voltage_pu(emt_trace, bus_id)
    ax.plot(dp_trace.time, dp_v, label="DP3Ph", lw=1.1)
    ax.plot(emt_trace.time, emt_v, label="EMT3Ph (1-cycle RMS)", lw=1.0, alpha=0.8)
    ax.set_ylabel(f"Bus {bus_id} / V (pu)")
    ax.grid(True, alpha=0.3)
    summary.append(
        (
            bus_id,
            dp_v[0],
            dp_v.min(),
            dp_v.max(),
            dp_v[-1],
            emt_v.dropna().iloc[0],
            emt_v.min(),
            emt_v.max(),
            emt_v.dropna().iloc[-1],
        )
    )
axes[0].legend(ncol=2)
axes[-1].set_xlabel("Time (s)")
fig.suptitle("IEEE118 bus voltage: DP3Ph vs EMT3Ph, 30-second SG4 runs")
fig.tight_layout()
plt.show()
summary_df = pd.DataFrame(
    summary,
    columns=[
        "bus",
        "DP start",
        "DP min",
        "DP max",
        "DP end",
        "EMT start",
        "EMT min",
        "EMT max",
        "EMT end",
    ],
)
display(summary_df.round(4))
print(
    "All logged DP and EMT voltages finite:",
    np.isfinite(dp_trace.iloc[:, 1:].to_numpy()).all()
    and np.isfinite(emt_trace.iloc[:, 1:].to_numpy()).all(),
)
for gen_bus in (10, 69, 89):
    dp_w = dp_trace[f"Gen_N{gen_bus}.w_r"].to_numpy()
    emt_w = emt_trace[f"Gen_N{gen_bus}.w_r"].to_numpy()
    print(
        f"Gen {gen_bus} speed bounds: DP {dp_w.min():.5f}–{dp_w.max():.5f} pu; EMT {emt_w.min():.5f}–{emt_w.max():.5f} pu"
    )

## Transient comparison: temporary load increase

A 10% resistive load pulse is applied at the largest-load bus from 1.0 to 1.5 seconds. DP3Ph and EMT3Ph use the same added load and generator models.

In [ ]:
LOAD_STEP_TIME = 1.0
LOAD_CLEAR_TIME = 1.5
TRANSIENT_FINAL_TIME = 5.0
LOAD_BUS = int(BUS[np.argmax(BUS[:, 2]), 0])
LOAD_STEP_MW = 0.10 * BUS[np.argmax(BUS[:, 2]), 2]
LOAD_STEP_RESISTANCE = (
    BUS[np.argmax(BUS[:, 2]), 9] * 1e3 * BUS[np.argmax(BUS[:, 2]), 7]
) ** 2 / (LOAD_STEP_MW * 1e6)
GEN_BUS = int(GEN[np.argmax(GEN[:, 8]), 0])


def run_load_pulse(domain, name, time_step):
    is_dp = domain == matpower.Domain.DP3PH
    reader = matpower.Reader(str(STATIC_PATH), mpc_dyn_file_path=str(DYNAMIC_PATH))
    reader.create_dpsim_objects(
        domain,
        FREQUENCY,
        log_level=dpsimpy.LogLevel.off,
        with_pss=False,
        with_avr=False,
        with_tg=False,
        filter_out_of_service=True,
        generator_model=matpower.GenModel.SG_4ORDER_VBR,
    )
    node = reader.dpsimpy_busses_dict[f"N{LOAD_BUS}"]
    phase_parameter = dpsimpy.Math.single_phase_parameter_to_three_phase
    switch = (DP3PH_SWITCH if is_dp else EMT_PH3_SWITCH)(
        "LoadPulse", dpsimpy.LogLevel.off
    )
    switch.set_parameters(
        phase_parameter(1e9), phase_parameter(LOAD_STEP_RESISTANCE), False
    )
    ground = dpsimpy.dp.SimNode.gnd if is_dp else dpsimpy.emt.SimNode.gnd
    switch.connect([ground, node])
    reader.dpsimpy_comp_dict["LoadPulse"] = [switch, [ground, node]]
    reader.create_dpsim_topology()
    reader.system.init_with_powerflow(
        systemPF=pf_reader.system,
        domain=dpsimpy.Domain.DP if is_dp else dpsimpy.Domain.EMT,
    )
    logger = dpsimpy.Logger(name, True, 2 if not is_dp else 1)
    logger.log_attribute(f"N{LOAD_BUS}.V", "v", node)
    logger.log_attribute(
        f"Gen_N{GEN_BUS}.w_r", "w_r", reader.dpsimpy_comp_dict[f"Gen_N{GEN_BUS}"][0]
    )
    sim = dpsimpy.Simulation(name)
    sim.set_system(reader.system)
    sim.set_domain(dpsimpy.Domain.DP if is_dp else dpsimpy.Domain.EMT)
    sim.set_solver(dpsimpy.Solver.MNA)
    sim.do_init_from_nodes_and_terminals(True)
    sim.do_system_matrix_recomputation(True)
    sim.set_time_step(time_step)
    sim.set_final_time(TRANSIENT_FINAL_TIME)
    sim.add_logger(logger)
    sim.add_event(dpsimpy.event.SwitchEvent3Ph(LOAD_STEP_TIME, switch, True))
    sim.add_event(dpsimpy.event.SwitchEvent3Ph(LOAD_CLEAR_TIME, switch, False))
    start = time.perf_counter()
    sim.run()
    runtime = time.perf_counter() - start
    trace = pd.read_csv(REPO / "logs" / f"{name}.csv", skipinitialspace=True)
    return trace, runtime


dp_step_trace, dp_step_runtime = run_load_pulse(
    matpower.Domain.DP3PH, "IEEE118_DP3Ph_LoadPulse", 1e-3
)
emt_step_trace, emt_step_runtime = run_load_pulse(
    matpower.Domain.EMT, "IEEE118_EMT3Ph_LoadPulse", 5e-4
)


def dp_event_voltage_pu(trace):
    re = trace[f"N{LOAD_BUS}.V_1.re"].to_numpy()
    im = trace[f"N{LOAD_BUS}.V_1.im"].to_numpy()
    base_kv = BUS[LOAD_BUS - 1, 9]
    return np.hypot(re, im) / (base_kv * 1e3 * np.sqrt(2 / 3))


def emt_event_voltage_pu(trace):
    phases = [f"N{LOAD_BUS}.V_{p}" for p in range(3)]
    phase_rms = np.sqrt((trace[phases].to_numpy() ** 2).mean(axis=1))
    base_kv = BUS[LOAD_BUS - 1, 9]
    phase_base = base_kv * 1e3 / np.sqrt(3)
    cycle = max(
        2,
        round(
            1.0 / FREQUENCY / float(np.median(np.diff(emt_step_trace.time.to_numpy())))
        ),
    )
    values = np.sqrt(
        pd.Series((phase_rms / phase_base) ** 2)
        .rolling(cycle, min_periods=cycle)
        .mean()
    )
    return values.to_numpy()


dp_step_voltage = dp_event_voltage_pu(dp_step_trace)
emt_step_voltage = emt_event_voltage_pu(emt_step_trace)
dp_step_speed = dp_step_trace[f"Gen_N{GEN_BUS}.w_r"].to_numpy()
emt_step_speed = emt_step_trace[f"Gen_N{GEN_BUS}.w_r"].to_numpy()
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(dp_step_trace.time, dp_step_voltage, label="DP3Ph", lw=1.0)
axes[0].plot(
    emt_step_trace.time, emt_step_voltage, label="EMT3Ph (1-cycle RMS)", lw=1.0
)
axes[0].set_ylabel(f"Bus {LOAD_BUS} voltage (pu)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[1].plot(dp_step_trace.time, dp_step_speed, label="DP3Ph", lw=1.0)
axes[1].plot(emt_step_trace.time, emt_step_speed, label="EMT3Ph", lw=1.0)
axes[1].axhline(1.0, color="black", ls="--", lw=0.8)
axes[1].set_ylabel(f"Generator {GEN_BUS} speed (pu)")
axes[1].set_xlabel("Time (s)")
axes[1].grid(True, alpha=0.3)
for ax in axes:
    ax.axvspan(LOAD_STEP_TIME, LOAD_CLEAR_TIME, color="orange", alpha=0.15)
fig.suptitle("IEEE118 response to the same temporary load increase")
fig.tight_layout()
display(fig)
plt.close(fig)
print(
    f"Load pulse: +{LOAD_STEP_MW:.1f} MW at bus {LOAD_BUS} from {LOAD_STEP_TIME:g} to {LOAD_CLEAR_TIME:g} s"
)
print(
    f"DP transient: {len(dp_step_trace)} samples, {dp_step_runtime:.2f} s; EMT: {len(emt_step_trace)} samples, {emt_step_runtime:.2f} s; EMT/DP={emt_step_runtime/dp_step_runtime:.2f}x"
)
print(
    f"Bus {LOAD_BUS} voltage bounds: DP {np.nanmin(dp_step_voltage):.4f}–{np.nanmax(dp_step_voltage):.4f} pu; EMT {np.nanmin(emt_step_voltage):.4f}–{np.nanmax(emt_step_voltage):.4f} pu"
)
print(
    f"Generator {GEN_BUS} maximum |speed−1|: DP {np.max(np.abs(dp_step_speed-1)):.5f} pu; EMT {np.max(np.abs(emt_step_speed-1)):.5f} pu; all transient traces finite={np.isfinite(dp_step_trace.iloc[:,1:].to_numpy()).all() and np.isfinite(emt_step_trace.iloc[:,1:].to_numpy()).all()}"
)